# Redrob Ranker — Full Pipeline Notebook (v3 GPU-precompute + validator CSV)

**Two phases:**

1. **PRECOMPUTE** — run once on your own machine. This can use GPU. It embeds survivor job texts and saves local artifacts:
   - `artifacts/job_embeddings.npy`
   - `artifacts/query_embeddings.npy`
   - `artifacts/job_meta.parquet`

2. **RANKING** — the reproducible submission path. This loads precomputed artifacts only, runs CPU-safe matrix scoring + deterministic multipliers, and outputs a validator-ready CSV.

> ⚠️ Challenge-safe separation: GPU/model/network can be used for precompute, but the final ranking step must be CPU-only, no network, no hosted APIs, and under the time budget.

**v3 fixes added:**

- CUDA-aware BGE embedding cell with batch-size fallback on GPU OOM.
- Skips model loading if embeddings already exist.
- Saves embeddings as `float32` to keep artifact size/memory reasonable.
- Ranking section loads `.npy` artifacts with `mmap_mode='r'` and does not call the model.
- Output now creates the required submission format exactly: `candidate_id,rank,score,reasoning`.
- Adds local validator checks for 100 rows, rank order, duplicate candidates, monotonic scores, and column order.

Existing v2 fixes retained:

- `description_confidence()` NaN-safe via `safe_text()`
- `candidate_base.summary` / `headline` merged in and used for sparse job descriptions
- Fixed `REFERENCE_DATE` for reproducibility
- Trap multiplier requires explicit column
- `df_cand` seeded from all `survivor_ids`
- `duration_months` recomputed after date cleanup
- Best-job selection uses geometric mean, not single-M max


In [1]:
import sys, torch

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Still on CPU kernel. Change notebook kernel to project .venv.")

Python: E:\Coding\Projects\redrob-ranker\.venv\Scripts\python.exe
Torch: 2.11.0+cu128
Torch CUDA: 12.8
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
import os, time, gc, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import gmean as scipy_gmean
import torch

# ─────────────────────────────────────────────────────────────────────────────
# Paths
# ─────────────────────────────────────────────────────────────────────────────
# Keep the Windows absolute DATA_DIR default for your local machine, but allow
# environment-variable overrides for GitHub/Colab/sandbox use.
PROJECT_ROOT = Path(os.getenv("REDROB_PROJECT_ROOT", Path.cwd()))
DATA_DIR = Path(os.getenv(
    "REDROB_DATA_DIR",
    r"E:\Coding\Projects\redrob-ranker\dataset\artifacts",
))
EMBED_DIR = Path(os.getenv("REDROB_EMBED_DIR", PROJECT_ROOT / "artifacts"))
OUTPUT_DIR = Path(os.getenv("REDROB_OUTPUT_DIR", PROJECT_ROOT / "output"))

EMBED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Change this before final upload, or set TEAM_ID in your shell.
# The output will be output/<TEAM_ID>.csv
TEAM_ID = os.getenv("TEAM_ID", "team_xxx")

# ─────────────────────────────────────────────────────────────────────────────
# Precompute device: GPU is allowed here. Ranking later does not call the model.
# ─────────────────────────────────────────────────────────────────────────────
FORCE_CPU_PRECOMPUTE = os.getenv("FORCE_CPU_PRECOMPUTE", "0") == "1"
PRECOMPUTE_DEVICE = "cuda" if (torch.cuda.is_available() and not FORCE_CPU_PRECOMPUTE) else "cpu"

if PRECOMPUTE_DEVICE == "cuda":
    # RTX cards usually tolerate 512 for bge-base; reduce via BGE_BATCH_SIZE if OOM.
    BATCH_SIZE = int(os.getenv("BGE_BATCH_SIZE", "512"))
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"Precompute device : CUDA — {torch.cuda.get_device_name(0)}")
    print(f"VRAM              : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"Embedding batch   : {BATCH_SIZE}")
else:
    BATCH_SIZE = int(os.getenv("BGE_BATCH_SIZE", "128"))
    print("Precompute device : CPU")
    print(f"Embedding batch   : {BATCH_SIZE}")

# Fixed reference date for reproducibility.
# Override via: set REFERENCE_DATE=2026-06-25
REFERENCE_DATE = pd.Timestamp(os.getenv("REFERENCE_DATE", "2026-06-25"))
QUERY_KEYS = ['M1','M2','M3','M4','N1','N2','N3','N4','IR','CV','HANDS_ON']
GMEAN_FLOOR = 0.05

print(f"\nProject root   : {PROJECT_ROOT}")
print(f"Data dir       : {DATA_DIR}")
print(f"Artifact dir   : {EMBED_DIR}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Team ID/file   : {TEAM_ID}.csv")
print(f"Reference date : {REFERENCE_DATE.date()}")


Precompute device : CUDA — NVIDIA GeForce RTX 4060 Laptop GPU
VRAM              : 8.6 GB
Embedding batch   : 512

Project root   : E:\Coding\Projects\redrob-ranker\notebooks
Data dir       : E:\Coding\Projects\redrob-ranker\dataset\artifacts
Artifact dir   : E:\Coding\Projects\redrob-ranker\notebooks\artifacts
Output dir     : E:\Coding\Projects\redrob-ranker\notebooks\output
Team ID/file   : team_xxx.csv
Reference date : 2026-06-25


## Part 1 — Data Loading

In [3]:
print("Working directory:", Path.cwd())
print("DATA_DIR exists?", DATA_DIR.exists(), "→", DATA_DIR)

required_artifacts = [
    'candidate_base.parquet',
    'candidate_jobs.parquet',
    'availability_scores.parquet',
    'survivors.parquet',
]
for name in required_artifacts:
    path = DATA_DIR / name
    print(f"{name:30s} {'✅' if path.exists() else '❌'} {path}")

missing = [name for name in required_artifacts if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        "Missing required parquet artifacts: " + ", ".join(missing) +
        "\nFix DATA_DIR or run earlier pipeline steps first."
    )


Working directory: E:\Coding\Projects\redrob-ranker\notebooks
DATA_DIR exists? True → E:\Coding\Projects\redrob-ranker\dataset\artifacts
candidate_base.parquet         ✅ E:\Coding\Projects\redrob-ranker\dataset\artifacts\candidate_base.parquet
candidate_jobs.parquet         ✅ E:\Coding\Projects\redrob-ranker\dataset\artifacts\candidate_jobs.parquet
availability_scores.parquet    ✅ E:\Coding\Projects\redrob-ranker\dataset\artifacts\availability_scores.parquet
survivors.parquet              ✅ E:\Coding\Projects\redrob-ranker\dataset\artifacts\survivors.parquet


In [4]:
df_base      = pd.read_parquet(DATA_DIR / 'candidate_base.parquet')
df_jobs_raw  = pd.read_parquet(DATA_DIR / 'candidate_jobs.parquet')
df_avail     = pd.read_parquet(DATA_DIR / 'availability_scores.parquet')
df_survivors = pd.read_parquet(DATA_DIR / 'survivors.parquet')

# ── Inspect columns — verify these match your parquets before proceeding ──
print("=== candidate_jobs columns ===")
print(df_jobs_raw.dtypes)
print()
print("=== candidate_base columns ===")
print(df_base.dtypes)

# ── Filter job rows to survivors only ─────────────────────────────────────
survivor_ids = set(df_survivors['candidate_id'])
df_jobs = df_jobs_raw[df_jobs_raw['candidate_id'].isin(survivor_ids)].copy()

print(f"\nSurvivors           : {len(survivor_ids):,}")
print(f"Job rows (raw)      : {len(df_jobs_raw):,}")
print(f"Job rows (survivors): {len(df_jobs):,}")
print(f"Avg jobs/candidate  : {len(df_jobs)/len(survivor_ids):.1f}")

# FIX: Track candidates with no job rows — they'll still appear in df_cand
cands_with_jobs = set(df_jobs['candidate_id'].unique())
no_job_cands    = survivor_ids - cands_with_jobs
print(f"Survivors with no job rows: {len(no_job_cands):,}  (will score 0 on all M)")


=== candidate_jobs columns ===
candidate_id         str
job_index          int64
company              str
title                str
start_date           str
end_date             str
duration_months    int64
is_current          bool
industry             str
company_size         str
description          str
dtype: object

=== candidate_base columns ===
candidate_id                str
anonymized_name             str
headline                    str
summary                     str
location                    str
country                     str
years_of_experience     float64
current_title               str
current_company             str
current_company_size        str
current_industry            str
dtype: object

Survivors           : 90,236
Job rows (raw)      : 300,171
Job rows (survivors): 285,346
Avg jobs/candidate  : 3.2
Survivors with no job rows: 0  (will score 0 on all M)


## Part 2 — Precompute Embeddings *(run once — output saved to disk)*

Produces:
- `artifacts/job_embeddings.npy` — shape `(N_survivor_jobs, 768)` ~920 MB
- `artifacts/job_meta.parquet`   — metadata needed at ranking time
- `artifacts/query_embeddings.npy` — shape `(11, 768)`

Once these files exist, skip to **Part 3**.


In [5]:
# ── Must-have queries (M1–M4) ─────────────────────────────────────────────
MUST_HAVE_QUERIES = {
    'M1': (
        'production deployment of embedding-based retrieval or semantic search '
        'systems to real users, handling embedding drift, index refresh, and '
        'retrieval quality regression in production'
    ),
    'M2': (
        'hands-on production experience with vector databases or hybrid search '
        'infrastructure such as FAISS, Pinecone, Weaviate, Qdrant, Milvus, '
        'OpenSearch, or Elasticsearch — operational experience not just tutorials'
    ),
    'M3': (
        'designing and operating ranking evaluation frameworks, NDCG, MRR, MAP, '
        'offline-to-online correlation, A/B testing of ranking or recommendation '
        'systems, understanding how to measure ranking quality rigorously'
    ),
    'M4': (
        'personally implemented and deployed ML or backend systems to production, '
        'wrote production Python code, owned deployed services, hands-on '
        'implementation rather than architecture oversight or team management'
    ),
}

# ── Nice-to-have queries (N1–N4) — bonus only, never penalises absence ────
NICE_TO_HAVE_QUERIES = {
    'N1': (
        'LLM fine-tuning experience — LoRA, QLoRA, PEFT, parameter-efficient '
        'fine-tuning, instruction tuning, RLHF'
    ),
    'N2': (
        'learning-to-rank models, XGBoost LTR, neural ranking, '
        'listwise or pairwise ranking losses, LambdaMART'
    ),
    'N3': (
        'HR technology, recruiting platform, marketplace product, '
        'talent intelligence, job matching, candidate search product'
    ),
    'N4': (
        'distributed systems, large-scale ML inference optimization, '
        'model serving at scale, latency optimization, GPU inference'
    ),
}

# ── Domain + hands-on queries ─────────────────────────────────────────────
DOMAIN_QUERIES = {
    'IR': (
        'information retrieval, NLP, natural language processing, search, ranking, '
        'recommendation systems, text embeddings, semantic search, '
        'retrieval augmented generation, question answering'
    ),
    'CV': (
        'computer vision, image recognition, object detection, speech recognition, '
        'audio processing, robotics, pose estimation, image segmentation, '
        'convolutional neural network'
    ),
}

HANDS_ON_QUERY = (
    'personally implemented and deployed ML or backend systems, '
    'wrote production code, built and shipped features to real users, '
    'hands-on engineering not team management or architecture review'
)

# Ordered query list — index order MUST match QUERY_KEYS
QUERY_TEXTS = [
    MUST_HAVE_QUERIES['M1'], MUST_HAVE_QUERIES['M2'],
    MUST_HAVE_QUERIES['M3'], MUST_HAVE_QUERIES['M4'],
    NICE_TO_HAVE_QUERIES['N1'], NICE_TO_HAVE_QUERIES['N2'],
    NICE_TO_HAVE_QUERIES['N3'], NICE_TO_HAVE_QUERIES['N4'],
    DOMAIN_QUERIES['IR'], DOMAIN_QUERIES['CV'],
    HANDS_ON_QUERY,
]
assert len(QUERY_KEYS) == 11 == len(QUERY_TEXTS), "Query count mismatch"
print("Query keys:", QUERY_KEYS)


Query keys: ['M1', 'M2', 'M3', 'M4', 'N1', 'N2', 'N3', 'N4', 'IR', 'CV', 'HANDS_ON']


In [6]:
# ── NaN-safe text helper (FIX 1) ──────────────────────────────────────────
def safe_text(x):
    """Return empty string for NaN/None; str() everything else."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    return str(x).strip()

# ── Tech anchors for confidence scoring ───────────────────────────────────
TECH_ANCHORS = {
    'milvus','pinecone','faiss','elasticsearch','qdrant','weaviate',
    'bert','embedding','vector','retrieval','ndcg','ranking','transformer',
    'fine-tun','pytorch','sklearn','xgboost','rag','llm','semantic',
    'reranking','bm25','dense','sparse','hybrid','sentence-transformer',
}

def description_confidence(text):
    """
    Scalar [0.20, 1.0] penalising vague short descriptions.
    Short-but-technical gets a floor of 0.75.
    FIX 1: uses safe_text() — NaN-safe, no crash.
    """
    text = safe_text(text)         # FIX: was: if not text or not text.strip()
    if not text:
        return 0.20
    wc = len(text.split())
    has_anchor = any(t in text.lower() for t in TECH_ANCHORS)
    if wc >= 30:
        return 1.0
    elif has_anchor:
        return max(0.75, min(1.0, wc / 30))
    else:
        return min(1.0, wc / 30)

def enrich_job_text(row):
    """
    Enrich job text for embedding.
    FIX 2: For sparse job descriptions (<15 words), fall back to candidate
    headline + summary from candidate_base (merged in earlier).
    Do NOT include company name — adds noise.
    """
    title       = safe_text(row.get('title'))
    description = safe_text(row.get('description'))
    industry    = safe_text(row.get('industry'))
    # Profile fields (present after merge with df_base)
    headline    = safe_text(row.get('headline'))
    summary     = safe_text(row.get('summary'))

    wc = len(description.split())

    if wc < 15:
        # Sparse job description — augment with candidate profile context
        # Cap summary at 300 chars so it doesn't swamp the job title
        profile_ctx = ' '.join(filter(None, [industry, headline, summary[:300]]))
        return f"{title}. {profile_ctx}. {description}".strip()
    else:
        return f"{title}. {description}".strip()

print("Functions defined: safe_text, description_confidence, enrich_job_text")


Functions defined: safe_text, description_confidence, enrich_job_text


In [7]:
# FIX 2: Merge candidate profile fields into df_jobs BEFORE enrichment ─────
# Secondary signal for sparse descriptions: headline, summary, current_title
PROFILE_COLS = [
    c for c in [
        'candidate_id', 'headline', 'summary', 'location',
        'country', 'years_of_experience', 'current_title',
    ]
    if c in df_base.columns
]
print(f"Profile columns available for merge: {PROFILE_COLS}")

df_jobs = df_jobs.merge(
    df_base[PROFILE_COLS].drop_duplicates('candidate_id'),
    on='candidate_id',
    how='left',
)
print(f"df_jobs shape after merge: {df_jobs.shape}")

# ── Apply text enrichment ─────────────────────────────────────────────────
df_jobs['enriched_text']          = df_jobs.apply(enrich_job_text, axis=1)
df_jobs['description_confidence'] = df_jobs['description'].apply(description_confidence)

print("\nConfidence distribution:")
print(df_jobs['description_confidence'].describe().round(3))
low_pct = (df_jobs['description_confidence'] < 0.75).mean() * 100
print(f"Low-confidence jobs (<0.75): {low_pct:.1f}%")

# ── Detect company column name ─────────────────────────────────────────────
company_col = next(
    (c for c in ['company','employer','organization','company_name'] if c in df_jobs.columns),
    None,
)
if company_col:
    print(f"\nCompany column found: '{company_col}'")
else:
    df_jobs['company'] = 'Unknown'
    company_col = 'company'
    print("\nNo company column found — defaulting to 'Unknown'")

# ── Fix date types ─────────────────────────────────────────────────────────
df_jobs['end_date']   = pd.to_datetime(df_jobs['end_date'],   errors='coerce')
df_jobs['start_date'] = pd.to_datetime(df_jobs['start_date'], errors='coerce')
df_jobs['is_current'] = df_jobs['is_current'].fillna(False).astype(bool)

# Fill NaT end_date for current roles
df_jobs.loc[df_jobs['is_current'], 'end_date'] = REFERENCE_DATE

# FIX 6: Always recompute duration_months after date cleanup.
# Do NOT reuse pre-existing values — current roles had stale end_dates before this cell.
df_jobs['duration_months'] = (
    (df_jobs['end_date'] - df_jobs['start_date']).dt.days / 30.44
).clip(lower=0)

df_jobs['months_since_end'] = (
    (REFERENCE_DATE - df_jobs['end_date']).dt.days / 30.44
).clip(lower=0)

# ── Save job_meta for ranking step ────────────────────────────────────────
META_COLS = [
    'candidate_id', 'title', company_col,
    'description_confidence', 'is_current',
    'end_date', 'duration_months', 'months_since_end',
]
df_meta_save = (
    df_jobs[META_COLS]
    .rename(columns={company_col: 'company'})
    .reset_index(drop=True)
)
df_meta_save.to_parquet(EMBED_DIR / 'job_meta.parquet', index=True)
print(f"\nSaved job_meta.parquet: {len(df_meta_save):,} rows → {EMBED_DIR / 'job_meta.parquet'}")


Profile columns available for merge: ['candidate_id', 'headline', 'summary', 'location', 'country', 'years_of_experience', 'current_title']
df_jobs shape after merge: (285346, 17)

Confidence distribution:
count    285346.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: description_confidence, dtype: float64
Low-confidence jobs (<0.75): 0.0%

Company column found: 'company'

Saved job_meta.parquet: 285,346 rows → E:\Coding\Projects\redrob-ranker\notebooks\artifacts\job_meta.parquet


In [8]:
from sentence_transformers import SentenceTransformer

JOB_EMB_PATH   = EMBED_DIR / 'job_embeddings.npy'
QUERY_EMB_PATH = EMBED_DIR / 'query_embeddings.npy'


def encode_with_oom_backoff(model, texts, *, batch_size, device, label):
    """
    Encode texts with normalized BGE embeddings.
    If CUDA OOM happens, halve the batch size and retry automatically.
    """
    current_batch = int(batch_size)
    while True:
        try:
            print(f"Encoding {len(texts):,} {label} on {device} (batch={current_batch}) ...")
            t0 = time.time()
            arr = model.encode(
                texts,
                batch_size=current_batch,
                normalize_embeddings=True,   # L2 norm → dot product == cosine similarity
                show_progress_bar=True,
                convert_to_numpy=True,
                device=device,
            )
            arr = np.asarray(arr, dtype=np.float32)
            elapsed = time.time() - t0
            print(f"Done in {elapsed/60:.1f} min — shape: {arr.shape}, dtype={arr.dtype}")
            return arr
        except RuntimeError as e:
            msg = str(e).lower()
            if device.startswith('cuda') and ('out of memory' in msg or 'cuda' in msg):
                if current_batch <= 32:
                    raise
                print(f"CUDA OOM at batch={current_batch}. Retrying with batch={current_batch // 2} ...")
                torch.cuda.empty_cache()
                gc.collect()
                current_batch //= 2
            else:
                raise

need_job_embeddings = not JOB_EMB_PATH.exists()
need_query_embeddings = not QUERY_EMB_PATH.exists()

if need_job_embeddings or need_query_embeddings:
    print("Loading BAAI/bge-base-en-v1.5 ...")
    model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=PRECOMPUTE_DEVICE)
    print("Model loaded.\n")
else:
    model = None
    print("Both embedding files already exist — skipping model load.")

# ── Encode job texts ───────────────────────────────────────────────────────
if JOB_EMB_PATH.exists():
    print(f"job_embeddings.npy already exists ({JOB_EMB_PATH.stat().st_size/1e6:.0f} MB).")
    print("Delete it if you intentionally want to re-embed job texts.")
else:
    texts = df_jobs['enriched_text'].fillna('').astype(str).tolist()
    job_embeddings = encode_with_oom_backoff(
        model,
        texts,
        batch_size=BATCH_SIZE,
        device=PRECOMPUTE_DEVICE,
        label='job texts',
    )
    np.save(JOB_EMB_PATH, job_embeddings)
    print(f"Saved → {JOB_EMB_PATH}  ({JOB_EMB_PATH.stat().st_size/1e6:.0f} MB)")
    del job_embeddings
    gc.collect()
    if PRECOMPUTE_DEVICE == 'cuda':
        torch.cuda.empty_cache()

# ── Encode query strings ───────────────────────────────────────────────────
if QUERY_EMB_PATH.exists():
    print(f"query_embeddings.npy already exists ({QUERY_EMB_PATH.stat().st_size/1e6:.2f} MB).")
else:
    query_embeddings = encode_with_oom_backoff(
        model,
        QUERY_TEXTS,
        batch_size=11,
        device=PRECOMPUTE_DEVICE,
        label='query strings',
    )
    np.save(QUERY_EMB_PATH, query_embeddings)
    print(f"Saved → {QUERY_EMB_PATH}  shape: {query_embeddings.shape}")
    del query_embeddings

# Release model memory before ranking cells.
if model is not None:
    del model
    gc.collect()
    if PRECOMPUTE_DEVICE == 'cuda':
        torch.cuda.empty_cache()
        print("Released CUDA cache after precompute.")


E:\Coding\Projects\redrob-ranker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading BAAI/bge-base-en-v1.5 ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6518.79it/s]


Model loaded.

Encoding 285,346 job texts on cuda (batch=512) ...


Batches: 100%|██████████| 558/558 [15:47<00:00,  1.70s/it]


Done in 15.8 min — shape: (285346, 768), dtype=float32
Saved → E:\Coding\Projects\redrob-ranker\notebooks\artifacts\job_embeddings.npy  (877 MB)
Encoding 11 query strings on cuda (batch=11) ...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.85it/s]


Done in 0.0 min — shape: (11, 768), dtype=float32
Saved → E:\Coding\Projects\redrob-ranker\notebooks\artifacts\query_embeddings.npy  shape: (11, 768)
Released CUDA cache after precompute.


---
## Part 3 — Ranking *(CPU-only reproducible path)*

This section loads pre-computed `.npy` / `.parquet` artifacts only.

**Important:** Do not call `SentenceTransformer`, hosted APIs, or any network service below this point. This is the part you should mirror in `rank.py` for Stage 3 reproduction.


In [9]:
# RANKING PHASE: CPU-safe, artifact-only.
# Uses NumPy matrix multiplication on precomputed embeddings. No model calls.
print("RANKING PHASE: loading local artifacts only — no model, no network, no GPU calls.")

for path in [JOB_EMB_PATH, QUERY_EMB_PATH, EMBED_DIR / 'job_meta.parquet']:
    if not path.exists():
        raise FileNotFoundError(f"Missing required ranking artifact: {path}")

# mmap keeps memory lower; matrix multiply still materializes only the small score matrix.
job_embeddings   = np.load(JOB_EMB_PATH, mmap_mode='r')
query_embeddings = np.asarray(np.load(QUERY_EMB_PATH), dtype=np.float32)
df_meta          = pd.read_parquet(EMBED_DIR / 'job_meta.parquet')

assert len(job_embeddings) == len(df_meta), (
    f"Row mismatch: embeddings={len(job_embeddings)}, meta={len(df_meta)}"
)
assert query_embeddings.shape[0] == len(QUERY_KEYS), (
    f"Query mismatch: embeddings={query_embeddings.shape[0]}, keys={len(QUERY_KEYS)}"
)

print(f"job_embeddings   : {job_embeddings.shape} / {job_embeddings.dtype}")
print(f"query_embeddings : {query_embeddings.shape} / {query_embeddings.dtype}")
print(f"job_meta rows    : {len(df_meta):,}")

# ── Matrix multiply: (N_jobs, 768) @ (768, 11) = (N_jobs, 11) ─────────────
t0 = time.time()
score_matrix = np.asarray(job_embeddings @ query_embeddings.T, dtype=np.float32)
print(f"\nScore matrix: {score_matrix.shape} computed in {time.time()-t0:.2f}s")

score_matrix = np.clip(score_matrix, 0, None)   # clamp negatives (model noise)

for i, key in enumerate(QUERY_KEYS):
    df_meta[f'score_{key}'] = score_matrix[:, i]

print("\nRaw M-score distributions:")
print(df_meta[[f'score_{k}' for k in ['M1','M2','M3','M4']]].describe().round(3))


RANKING PHASE: loading local artifacts only — no model, no network, no GPU calls.
job_embeddings   : (285346, 768) / float32
query_embeddings : (11, 768) / float32
job_meta rows    : 285,346

Score matrix: (285346, 11) computed in 0.23s

Raw M-score distributions:
         score_M1    score_M2    score_M3    score_M4
count  285346.000  285346.000  285346.000  285346.000
mean        0.469       0.532       0.491       0.542
std         0.049       0.054       0.039       0.048
min         0.379       0.408       0.383       0.445
25%         0.433       0.490       0.466       0.505
50%         0.461       0.535       0.493       0.534
75%         0.508       0.571       0.518       0.570
max         0.781       0.752       0.731       0.690


In [10]:
# ── Must-have recency (steep) ─────────────────────────────────────────────
df_meta['recency_mh'] = np.select(
    condlist=[
        df_meta['is_current'],
        df_meta['months_since_end'] <= 18,
        df_meta['months_since_end'] <= 36,
        df_meta['months_since_end'] <= 60,
    ],
    choicelist=[1.25, 1.00, 0.65, 0.35],
    default=0.15,
)

# ── Nice-to-have recency (gentle) ────────────────────────────────────────
df_meta['recency_nth'] = np.select(
    condlist=[
        df_meta['is_current'],
        df_meta['months_since_end'] <= 36,
        df_meta['months_since_end'] <= 60,
    ],
    choicelist=[1.15, 1.00, 0.75],
    default=0.50,
)

print("Recency weight distributions:")
print(df_meta[['recency_mh','recency_nth']].describe().round(3))
print(f"\nCurrent-job rows: {df_meta['is_current'].sum():,}")


Recency weight distributions:
       recency_mh  recency_nth
count  285346.000   285346.000
mean        0.637        0.841
std         0.464        0.270
min         0.150        0.500
25%         0.150        0.500
50%         0.350        0.750
75%         1.250        1.150
max         1.250        1.150

Current-job rows: 90,236


In [11]:
# ── Per-job adjusted M scores ─────────────────────────────────────────────
M_COLS = ['M1','M2','M3','M4']
for m in M_COLS:
    df_meta[f'adj_{m}'] = (
        df_meta[f'score_{m}']
        * df_meta['description_confidence']
        * df_meta['recency_mh']
    ).clip(0, 1.0)

# FIX 5: Seed df_cand from ALL survivor_ids, not just those with job rows.
# Candidates with no jobs get M scores of 0 (will naturally fall outside top-100).
df_cand = pd.DataFrame(
    index=pd.Index(sorted(survivor_ids), name='candidate_id')
)

recency_mh_sum = df_meta.groupby('candidate_id')['recency_mh'].sum()

# ── Per-candidate per-M: 70/30 blend of best-job + recency-weighted avg ────
for m in M_COLS:
    best  = df_meta.groupby('candidate_id')[f'adj_{m}'].max()
    w_avg = (
        df_meta.groupby('candidate_id')[f'adj_{m}'].sum() / recency_mh_sum
    ).fillna(0)
    df_cand[f'{m}_score'] = (
        (0.70 * best + 0.30 * w_avg)
        .reindex(df_cand.index)   # aligns to all survivors, NaN for no-job candidates
        .fillna(0)
        .clip(0, 1.0)
    )

print("Per-candidate M-score distributions:")
print(df_cand[['M1_score','M2_score','M3_score','M4_score']].describe().round(3))
print(f"\nZero-score M1 candidates: {(df_cand['M1_score'] == 0).sum():,}")


Per-candidate M-score distributions:
        M1_score   M2_score   M3_score   M4_score
count  90236.000  90236.000  90236.000  90236.000
mean       0.554      0.631      0.579      0.642
std        0.054      0.062      0.044      0.056
min        0.445      0.479      0.450      0.523
25%        0.513      0.585      0.552      0.599
50%        0.543      0.631      0.580      0.634
75%        0.596      0.671      0.608      0.677
max        0.911      0.879      0.849      0.811

Zero-score M1 candidates: 0


In [12]:
# ── Geometric mean of M1–M4 → must_have_score ─────────────────────────────
m_arr = df_cand[['M1_score','M2_score','M3_score','M4_score']].values.clip(GMEAN_FLOOR, 1.0)
df_cand['must_have_score'] = scipy_gmean(m_arr, axis=1)

print("must_have_score distribution:")
print(df_cand['must_have_score'].describe().round(3))
q = df_cand['must_have_score'].quantile([0.50, 0.75, 0.90, 0.95, 0.99])
print("\nKey quantiles:")
print(q.round(3).to_string())
if df_cand['must_have_score'].median() > 0.5:
    print("\n⚠️  WARNING: median > 0.5 — scoring is too generous")
else:
    print("\n✅  Right-skewed distribution as expected")


must_have_score distribution:
count    90236.000
mean         0.600
std          0.048
min          0.488
25%          0.563
50%          0.601
75%          0.629
max          0.794
Name: must_have_score, dtype: float64

Key quantiles:
0.50    0.601
0.75    0.629
0.90    0.669
0.95    0.683
0.99    0.720

⚠️  WARNING: median > 0.5 — scoring is too generous


In [13]:
N_COLS = ['N1','N2','N3','N4']
for n in N_COLS:
    df_meta[f'adj_{n}'] = (
        df_meta[f'score_{n}']
        * df_meta['description_confidence']
        * df_meta['recency_nth']
    ).clip(0, 1.0)

df_meta['best_nth'] = df_meta[[f'adj_{n}' for n in N_COLS]].max(axis=1)

recency_nth_sum = df_meta.groupby('candidate_id')['recency_nth'].sum()
df_cand['nth_raw'] = (
    df_meta.groupby('candidate_id')['best_nth'].sum() / recency_nth_sum
).reindex(df_cand.index).fillna(0).clip(0, 1.0)

df_cand['nth_bonus'] = (df_cand['nth_raw'] * 0.20).clip(0, 0.20)

print("Nice-to-have bonus distribution:")
print(df_cand['nth_bonus'].describe().round(3))


Nice-to-have bonus distribution:
count    90236.000
mean         0.112
std          0.006
min          0.093
25%          0.108
50%          0.111
75%          0.115
max          0.144
Name: nth_bonus, dtype: float64


In [14]:
# ── READ THESE before touching threshold constants in the next cell ────────
print("IR score quantiles (candidate-level max):")
cand_ir = df_meta.groupby('candidate_id')['score_IR'].max().reindex(df_cand.index).fillna(0)
print(cand_ir.quantile([0.25, 0.50, 0.75, 0.90]).round(3).to_dict())

print("\nCV score quantiles (candidate-level max):")
cand_cv = df_meta.groupby('candidate_id')['score_CV'].max().reindex(df_cand.index).fillna(0)
print(cand_cv.quantile([0.25, 0.50, 0.75, 0.90]).round(3).to_dict())

print("\nHANDS_ON score quantiles (candidate-level max):")
cand_ho_all = df_meta.groupby('candidate_id')['score_HANDS_ON'].max().reindex(df_cand.index).fillna(0)
print(cand_ho_all.quantile([0.25, 0.50, 0.75, 0.90]).round(3).to_dict())

print("\n→ Adjust threshold constants in the next cell based on these distributions.")


IR score quantiles (candidate-level max):
{0.25: 0.497, 0.5: 0.532, 0.75: 0.594, 0.9: 0.628}

CV score quantiles (candidate-level max):
{0.25: 0.508, 0.5: 0.539, 0.75: 0.55, 0.9: 0.561}

HANDS_ON score quantiles (candidate-level max):
{0.25: 0.565, 0.5: 0.593, 0.75: 0.624, 0.9: 0.642}

→ Adjust threshold constants in the next cell based on these distributions.


In [15]:
# ══════════════════════════════════════════════════════════════════════════
# CALIBRATE these thresholds after reviewing the quantile output above.
# ══════════════════════════════════════════════════════════════════════════

# ── Domain multiplier ──────────────────────────────────────────────────────
CV_HIGH_THRESH = 0.65
CV_MED_THRESH  = 0.50
IR_LOW_THRESH  = 0.40
IR_MED_THRESH  = 0.50

def domain_mult_fn(ir, cv):
    if cv > CV_HIGH_THRESH and ir < IR_LOW_THRESH:
        return 0.65
    elif cv > CV_MED_THRESH and ir < IR_MED_THRESH:
        return 0.90
    return 1.00

df_cand['domain_multiplier'] = [
    domain_mult_fn(cand_ir.get(cid, 0), cand_cv.get(cid, 0))
    for cid in df_cand.index
]
print("Domain multiplier distribution:")
print(df_cand['domain_multiplier'].value_counts().sort_index())

# ── Hands-on multiplier ────────────────────────────────────────────────────
recent_mask = df_meta['is_current'] | (df_meta['months_since_end'] <= 18)
recent_ho   = (
    df_meta[recent_mask]
    .groupby('candidate_id')['score_HANDS_ON']
    .max()
    .reindex(df_cand.index)
    .fillna(0)
)

HO_THRESHOLDS = [(0.75, 1.00), (0.55, 0.85), (0.35, 0.70)]

def hands_on_mult_fn(score):
    for thresh, mult in HO_THRESHOLDS:
        if score >= thresh:
            return mult
    return 0.55

df_cand['hands_on_multiplier'] = recent_ho.map(hands_on_mult_fn)
print("\nHands-on multiplier distribution:")
print(df_cand['hands_on_multiplier'].value_counts().sort_index())

# ── Job-hopper multiplier (deterministic, no embedding) ───────────────────
completed = df_meta[~df_meta['is_current']].copy()

median_tenure = completed.groupby('candidate_id')['duration_months'].median()
last3_avg = (
    completed.sort_values('months_since_end')
             .groupby('candidate_id')
             .head(3)
             .groupby('candidate_id')['duration_months']
             .mean()
)
current_dur = (
    df_meta[df_meta['is_current']]
    .groupby('candidate_id')['duration_months']
    .max()
)

def hopper_mult_fn(med, last3, cur):
    if   med >= 24 and last3 >= 18:  mult = 1.00
    elif med >= 18 or  last3 >= 18:  mult = 0.92
    elif med >= 12:                   mult = 0.82
    else:                             mult = 0.70
    if cur < 6:
        mult *= 0.95
    return mult

df_cand['job_hopper_multiplier'] = [
    hopper_mult_fn(
        median_tenure.get(cid, 24),
        last3_avg.get(cid, 24),
        current_dur.get(cid, 24),
    )
    for cid in df_cand.index
]
print("\nJob-hopper multiplier distribution:")
print(df_cand['job_hopper_multiplier'].value_counts().sort_index())


Domain multiplier distribution:
domain_multiplier
0.9    14598
1.0    75638
Name: count, dtype: int64

Hands-on multiplier distribution:
hands_on_multiplier
0.70    33114
0.85    57122
Name: count, dtype: int64

Job-hopper multiplier distribution:
job_hopper_multiplier
0.70     5150
0.82     7930
0.92    20995
1.00    56161
Name: count, dtype: int64


In [16]:
# ── Availability ──────────────────────────────────────────────────────────
avail = df_avail.set_index('candidate_id')['availability_multiplier']
df_cand['availability_multiplier'] = df_cand.index.map(avail).fillna(1.0)

# FIX 4: Trap multiplier — require explicit column, no boolean auto-detection ─
TRAP_PATH = DATA_DIR / 'trap_scores.parquet'
if TRAP_PATH.exists():
    df_trap = pd.read_parquet(TRAP_PATH).set_index('candidate_id')

    if 'trap_multiplier' not in df_trap.columns:
        raise ValueError(
            "trap_scores.parquet exists but has no 'trap_multiplier' column.\n"
            f"Columns found: {list(df_trap.columns)}\n"
            "Rename the correct column to 'trap_multiplier' (float, 0.0–1.0) before continuing."
        )

    df_cand['trap_multiplier'] = (
        df_cand.index.map(df_trap['trap_multiplier']).fillna(1.0)
    )
    print(f"Trap multiplier loaded. Distribution:")
    print(df_cand['trap_multiplier'].describe().round(3))
else:
    df_cand['trap_multiplier'] = 1.0
    print("Step 4 artifact not found — trap_multiplier = 1.0  (run Step 4 before final submission)")

# ── Final score assembly (multiplicative) ─────────────────────────────────
df_cand['semantic_score'] = (
    df_cand['must_have_score']
    * (1 + df_cand['nth_bonus'])
    * df_cand['domain_multiplier']
    * df_cand['hands_on_multiplier']
)

df_cand['final_score_raw'] = (
    df_cand['semantic_score']
    * df_cand['availability_multiplier']
    * df_cand['job_hopper_multiplier']
    * df_cand['trap_multiplier']
)

# ALWAYS rank on raw — floor is ONLY for the display column
df_cand['final_score'] = df_cand['final_score_raw'].clip(lower=0.10)

print("\nFinal score (raw) distribution:")
print(df_cand['final_score_raw'].describe().round(4))


Step 4 artifact not found — trap_multiplier = 1.0  (run Step 4 before final submission)

Final score (raw) distribution:
count    90236.0000
mean         0.2121
std          0.1023
min          0.0244
25%          0.1258
50%          0.2087
75%          0.2883
max          0.7134
Name: final_score_raw, dtype: float64


## Part 4 — Output

In [17]:
# FIX 7: Best matching job uses geometric mean of adj M scores, not single-M max.
# A job that only matches M1 should not be chosen over a balanced one.
df_meta['composite_M'] = scipy_gmean(
    df_meta[[f'adj_{m}' for m in M_COLS]].clip(GMEAN_FLOOR, 1.0),
    axis=1,
)
best_job = (
    df_meta.sort_values('composite_M', ascending=False)
           .groupby('candidate_id')
           .first()
           [['title','company','description_confidence']]
           .rename(columns={
               'title':                  'best_matching_job_title',
               'company':                'best_matching_job_company',
               'description_confidence': 'best_job_confidence',
           })
)
df_cand = df_cand.join(best_job)
df_cand['best_matching_job_title']   = df_cand['best_matching_job_title'].fillna('Unknown')
df_cand['best_matching_job_company'] = df_cand['best_matching_job_company'].fillna('Unknown')
df_cand['best_job_confidence']       = df_cand['best_job_confidence'].fillna(0.0)

# ── Confidence flags ───────────────────────────────────────────────────────
df_cand['low_description_confidence'] = df_cand['best_job_confidence'] < 0.75
all_sparse = df_meta.groupby('candidate_id')['description_confidence'].max().lt(0.75)
df_cand['all_sparse'] = df_cand.index.map(all_sparse).fillna(True)

# ── Reasoning string builder ───────────────────────────────────────────────
def build_reason(row):
    strengths = []
    if row['M1_score'] > 0.60: strengths.append('retrieval/search systems')
    if row['M2_score'] > 0.60: strengths.append('vector DB / hybrid search')
    if row['M3_score'] > 0.60: strengths.append('ranking evaluation frameworks')
    if row['M4_score'] > 0.60: strengths.append('hands-on production coding')

    flags = []
    if row['job_hopper_multiplier']   < 0.85: flags.append('job stability concern')
    if row['domain_multiplier']       < 1.00: flags.append('partial domain mismatch')
    if row['low_description_confidence']:     flags.append('limited description detail')
    if row['availability_multiplier'] < 0.80: flags.append('availability friction')
    if row.get('all_sparse', False):          flags.append('all jobs have sparse descriptions')
    if row['trap_multiplier']         < 1.00: flags.append('possible honeypot/profile inconsistency')

    strength_str = ', '.join(strengths) if strengths else 'general ML background'
    flag_str     = ('. Concern: ' + '; '.join(flags)) if flags else ''
    return (
        f"Evidence of {strength_str} from "
        f"{row['best_matching_job_title']} at {row['best_matching_job_company']} "
        f"with balanced must-have scores "
        f"M1:{row['M1_score']:.2f}, M2:{row['M2_score']:.2f}, "
        f"M3:{row['M3_score']:.2f}, M4:{row['M4_score']:.2f}{flag_str}."
    )

# ── Deterministic Top-100 ──────────────────────────────────────────────────
# Tie-break: candidate_id ascending, as allowed by the spec.
top100 = (
    df_cand.sort_values(['final_score_raw'], ascending=[False], kind='mergesort')
           .reset_index()
           .sort_values(['final_score_raw', 'candidate_id'], ascending=[False, True], kind='mergesort')
           .head(100)
           .copy()
)

top100['rank'] = np.arange(1, len(top100) + 1)
top100['reasoning'] = top100.apply(build_reason, axis=1)

# Diagnostic output: useful for debugging, not the portal upload.
DIAG_COLS = [
    'candidate_id', 'rank', 'final_score_raw', 'final_score', 'semantic_score', 'must_have_score',
    'M1_score', 'M2_score', 'M3_score', 'M4_score', 'nth_bonus',
    'domain_multiplier', 'hands_on_multiplier', 'job_hopper_multiplier',
    'availability_multiplier', 'trap_multiplier',
    'best_matching_job_title', 'best_matching_job_company',
    'low_description_confidence', 'all_sparse', 'reasoning',
]
diag = top100[DIAG_COLS].copy()
DIAG_PATH = OUTPUT_DIR / 'top100_ranked_diagnostic.csv'
diag.to_csv(DIAG_PATH, index=False, encoding='utf-8')

# Portal upload output: exact required columns/order.
submission = pd.DataFrame({
    'candidate_id': top100['candidate_id'].astype(str).values,
    'rank': top100['rank'].astype(int).values,
    # round for neatness; ties are allowed, ranks remain unique
    'score': top100['final_score_raw'].astype(float).round(6).values,
    'reasoning': top100['reasoning'].astype(str).values,
})
SUBMISSION_PATH = OUTPUT_DIR / f'{TEAM_ID}.csv'
submission.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8')

print(f"Saved validator-ready submission: {SUBMISSION_PATH}")
print(f"Saved diagnostic CSV          : {DIAG_PATH}")
print("\nTop 10 snapshot:")
print(
    submission.merge(
        top100[['candidate_id','must_have_score','best_matching_job_title']],
        on='candidate_id',
        how='left',
    )[['rank','candidate_id','score','must_have_score','best_matching_job_title']]
    .head(10)
    .to_string(index=False)
)


Saved validator-ready submission: E:\Coding\Projects\redrob-ranker\notebooks\output\team_xxx.csv
Saved diagnostic CSV          : E:\Coding\Projects\redrob-ranker\notebooks\output\top100_ranked_diagnostic.csv

Top 10 snapshot:
 rank candidate_id    score  must_have_score          best_matching_job_title
    1 CAND_0002025 0.713351         0.777537               Senior AI Engineer
    2 CAND_0043860 0.678342         0.739380               Junior ML Engineer
    3 CAND_0046132 0.660063         0.731811             AI Research Engineer
    4 CAND_0054394 0.641302         0.785166  Recommendation Systems Engineer
    5 CAND_0081846 0.637755         0.793982                 Lead AI Engineer
    6 CAND_0045887 0.635443         0.692188                   Data Scientist
    7 CAND_0033445 0.634697         0.700906                      ML Engineer
    8 CAND_0018499 0.629854         0.785715 Senior Machine Learning Engineer
    9 CAND_0011687 0.625064         0.765270              Senior NLP Eng

## Part 5 — Validation *(run before submitting)*

In [18]:
print("=" * 70)
print("VALIDATION CHECKS")
print("=" * 70)

# [0] Submission-format validator
print("\n[0] Submission CSV format:")
expected_cols = ['candidate_id', 'rank', 'score', 'reasoning']
sub = pd.read_csv(SUBMISSION_PATH)

assert list(sub.columns) == expected_cols, f"Column order wrong: {list(sub.columns)}"
assert len(sub) == 100, f"Expected 100 rows, got {len(sub)}"
assert sub['rank'].tolist() == list(range(1, 101)), "Ranks must be exactly 1..100 in order"
assert sub['candidate_id'].is_unique, "Duplicate candidate_id values found"
assert sub['rank'].is_unique, "Duplicate rank values found"
assert sub['score'].notna().all(), "NaN score found"
assert sub['reasoning'].notna().all(), "NaN reasoning found"
assert sub['score'].is_monotonic_decreasing, "Scores must be non-increasing with rank"

# Candidate ID existence check against known released base file.
if 'candidate_id' in df_base.columns:
    missing_ids = set(sub['candidate_id']) - set(df_base['candidate_id'])
    assert not missing_ids, f"Submission contains candidate_ids not in candidate_base: {list(missing_ids)[:5]}"

print("✅ Required columns/order OK")
print("✅ Exactly 100 rows")
print("✅ Ranks 1..100 OK")
print("✅ Candidate IDs unique and valid")
print("✅ Scores monotonic non-increasing")
print(f"✅ Ready CSV: {SUBMISSION_PATH}")

# [1] Score distribution
print("\n[1] must_have_score distribution:")
q = df_cand['must_have_score'].quantile([0.50, 0.75, 0.90, 0.95, 0.99])
print(q.round(3).to_string())
if df_cand['must_have_score'].median() > 0.50:
    print("⚠️  WARNING: Median > 0.5 — scoring may be too generous")
else:
    print("✅  Median < 0.5 — right-skewed as expected")

# [2] M1/M2 correlation
print("\n[2] M1/M2 Pearson correlation:")
corr = df_cand[['M1_score','M2_score']].corr().iloc[0,1]
print(f"    r = {corr:.3f}")
if corr > 0.90:
    print("⚠️  WARNING: >0.90 — M1/M2 redundant, consider averaging before gmean")
elif corr < 0.50:
    print("⚠️  WARNING: <0.50 — unexpectedly low, check query strings")
else:
    print("✅  In expected range")

# [3] Domain distributions
print("\n[3] Domain score quantiles:")
print("IR:", cand_ir.quantile([0.25,.50,.75,.90]).round(3).to_dict())
print("CV:", cand_cv.quantile([0.25,.50,.75,.90]).round(3).to_dict())

# [4] Floor check
floored = (df_cand['final_score'] == 0.10).sum()
print(f"\n[4] Candidates floored at 0.10: {floored:,} ({floored/len(df_cand)*100:.1f}%)")
if floored > len(df_cand) * 0.10:
    print("⚠️  WARNING: >10% floored — check if multipliers are too aggressive")
else:
    print("✅  Floor rate looks fine")

# [5] Honeypot check
print("\n[5] Honeypot/trap check:")
if 'trap_multiplier' in top100.columns:
    trap_count = int((top100['trap_multiplier'] < 1.0).sum())
    print(f"Trap candidates in top 100: {trap_count}")
    if trap_count >= 10:
        print("⚠️  WARNING: ≥10 honeypots in top 100 — likely disqualification risk")
    elif trap_count > 0:
        print("⚠️  Some trap candidates remain; inspect diagnostic CSV")
    else:
        print("✅  No trap candidates in top 100")
else:
    print("⚠️  trap_multiplier column missing")

# [6] Reasoning quality spot-check
print("\n[6] Top-20 spot-check:")
print(
    top100[['rank','candidate_id','final_score_raw','M1_score','M2_score',
            'M3_score','M4_score','best_matching_job_title']]
    .head(20)
    .to_string(index=False)
)

print("\n✅ Validation complete. Fix any warnings before final submission.")


VALIDATION CHECKS

[0] Submission CSV format:
✅ Required columns/order OK
✅ Exactly 100 rows
✅ Ranks 1..100 OK
✅ Candidate IDs unique and valid
✅ Scores monotonic non-increasing
✅ Ready CSV: E:\Coding\Projects\redrob-ranker\notebooks\output\team_xxx.csv

[1] must_have_score distribution:
0.50    0.601
0.75    0.629
0.90    0.669
0.95    0.683
0.99    0.720
⚠️  WARNING: Median > 0.5 — scoring may be too generous

[2] M1/M2 Pearson correlation:
    r = 0.769
✅  In expected range

[3] Domain score quantiles:
IR: {0.25: 0.497, 0.5: 0.532, 0.75: 0.594, 0.9: 0.628}
CV: {0.25: 0.508, 0.5: 0.539, 0.75: 0.55, 0.9: 0.561}

[4] Candidates floored at 0.10: 15,299 (17.0%)
⚠️  WARNING: >10% floored — check if multipliers are too aggressive

[5] Honeypot/trap check:
Trap candidates in top 100: 0
✅  No trap candidates in top 100

[6] Top-20 spot-check:
 rank candidate_id  final_score_raw  M1_score  M2_score  M3_score  M4_score          best_matching_job_title
    1 CAND_0002025         0.713351  0.801

## Final repo note to copy into README

Use this wording in your README so the compute split is defensible:

```md
Precompute step:
python precompute_embeddings.py --candidates ./dataset/candidates.jsonl --out ./artifacts

This step builds local embedding artifacts and may use GPU if available. It is not the reproduced ranking step.

Final ranking step:
python rank.py --candidates ./dataset/candidates.jsonl --artifacts ./artifacts --out ./output/team_xxx.csv

The final ranking step loads precomputed artifacts only, uses CPU/NumPy/Pandas, performs no network calls, and writes a CSV with columns candidate_id,rank,score,reasoning.
```


## Before upload checklist

- [ ] Rename `TEAM_ID` from `team_xxx` to your registered participant/team ID.
- [ ] Confirm `output/<TEAM_ID>.csv` has exactly: `candidate_id,rank,score,reasoning`.
- [ ] Run the validation cell with no assertion errors.
- [ ] Push notebook/code + artifacts or artifact-generation scripts to GitHub.
- [ ] Add README reproduction command.
- [ ] Add sandbox/demo link, e.g. Colab small-sample notebook.
- [ ] Export deck/PPT as PDF.
